In [2]:
# ==============================================================================
# GABUNGAN KODE UTUH NOTEBOOK KLASIFIKASI (FIXED & PENGECEKAN DIREKTORI OTOMATIS)
# ==============================================================================

import pandas as pd
import numpy as np
import os # Digunakan untuk melacak jalur folder secara dinamis demi menghindari FileNotFoundError
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import LabelEncoder
import joblib

# 1. Gunakan dataset hasil clustering yang memiliki fitur Target dengan Pengecekan Lokasi Bertingkat
file_target = 'data_clustering_inverse.csv'

if os.path.exists(file_target):
    df_class = pd.read_csv(file_target)
    print(f"[SUKSES] Berhasil memuat file '{file_target}' langsung dari folder saat ini.")
elif os.path.exists(os.path.join('BMLP_Wildan', file_target)):
    df_class = pd.read_csv(os.path.join('BMLP_Wildan', file_target))
    print(f"[SUKSES] Berhasil memuat file '{file_target}' dari dalam subfolder 'BMLP_Wildan'.")
else:
    # Menggunakan pencarian jalur absolut jika VS Code menggunakan root direktori yang berbeda
    base_dir = os.path.dirname(os.path.abspath('__file__'))
    potential_path = os.path.join(base_dir, file_target)
    if os.path.exists(potential_path):
        df_class = pd.read_csv(potential_path)
        print(f"[SUKSES] Berhasil memuat file melalui penelusuran jalur absolut.")
    else:
        # Jika file benar-benar tidak ada di folder mana pun
        raise FileNotFoundError(f"File '{file_target}' tidak dapat ditemukan! "
                                f"Pastikan Anda sudah menjalankan Notebook Clustering hingga cell paling akhir "
                                f"agar file .csv ini terbentuk di dalam folder Anda.")

# 2. Tampilkan 5 baris pertama dengan function head
print("\n=== LIMA BARIS PERTAMA DATA KLASIFIKASI ===")
print(df_class.head())


# === MULAI CODE OPSIONAL ===

# Drop kolom string tanggal atau sisa ID jika masih terbawa agar tidak mengacaukan model
drop_cols_additional = ['TransactionID', 'AccountID', 'DeviceID', 'IP Address', 'MerchantID', 'TransactionDate', 'PreviousTransactionDate']
for col in drop_cols_additional:
    if col in df_class.columns:
        df_class = df_class.drop(columns=[col])

# Preprocessing Encoding untuk fitur string/objek pasca-inverse agar terbaca model Scikit-Learn
for col in df_class.select_dtypes(include=['object']).columns:
    if col != 'Target':
        df_class[col] = LabelEncoder().fit_transform(df_class[col].astype(str))

X = df_class.drop(columns=['Target'])
y = df_class['Target']

# 3. Menggunakan train_test_split() untuk melakukan pembagian dataset.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 4. Buatlah model klasifikasi menggunakan Decision Tree
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train, y_train)

# 5. Menyimpan Model Decision Tree (Mandatory)
joblib.dump(dt_model, 'decision_tree_model.h5')

# 6. Melatih model menggunakan algoritma klasifikasi scikit-learn selain Decision Tree (Random Forest)
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)

# 7. Menampilkan hasil evaluasi akurasi, presisi, recall, dan F1-Score pada seluruh algoritma
def evaluasi(model, X_t, y_t, nama):
    preds = model.predict(X_t)
    print(f"\n--- Evaluasi Kontrol Kinerja: {nama} ---")
    print(f"Accuracy : {accuracy_score(y_t, preds):.4f}")
    print(f"Precision: {precision_score(y_t, preds, average='macro'):.4f}")
    print(f"Recall   : {recall_score(y_t, preds, average='macro'):.4f}")
    print(f"F1-Score : {f1_score(y_t, preds, average='macro'):.4f}")

evaluasi(dt_model, X_test, y_test, "Decision Tree (Baseline)")
evaluasi(rf_model, X_test, y_test, "Random Forest (Explore)")

# 8. Menyimpan Model Selain Decision Tree (Format wajib sesuai checklist)
joblib.dump(rf_model, 'explore_RandomForest_classification.h5')

# 9. Lakukan Hyperparameter Tuning dan Latih ulang.
param_grid = {'max_depth': [5, 10, None], 'criterion': ['gini', 'entropy']}
grid_search = GridSearchCV(DecisionTreeClassifier(random_state=42), param_grid, cv=3)
grid_search.fit(X_train, y_train)

# 10. Menampilkan hasil evaluasi akurasi, presisi, recall, dan F1-Score pada algoritma yang sudah dituning.
tuned_model = grid_search.best_estimator_
evaluasi(tuned_model, X_test, y_test, "Decision Tree (Tuned)")

# 11. Menyimpan Model hasil tuning
joblib.dump(tuned_model, 'tuning_classification.h5')

print("\n[SUKSES] Seluruh proses klasifikasi selesai dan model .h5 berhasil diekspor!")

[SUKSES] Berhasil memuat file 'data_clustering_inverse.csv' langsung dari folder saat ini.

=== LIMA BARIS PERTAMA DATA KLASIFIKASI ===
   TransactionAmount  TransactionType  Location   Channel  CustomerAge  \
0          -1.107930              0.0  1.200869 -1.088615     1.444427   
1           0.579431              0.0 -0.503358 -1.088615     1.331694   
2          -0.585158              0.0  0.145871  1.361177    -1.430270   
3          -0.313941              0.0  0.957408  1.361177    -1.035703   
4          -0.744226              0.0  0.551640 -1.088615    -1.486636   

   CustomerOccupation  TransactionDuration  LoginAttempts  AccountBalance  \
0           -1.316889            -0.537894            0.0        0.014221   
1           -1.316889             0.304662            0.0        2.253155   
2            1.311635            -0.888959            0.0       -1.018894   
3            1.311635            -1.324280            0.0        0.909321   
4            1.311635             